# Proof of Concept (PoC): 5-Tier Layered Enterprise Architecture — Credit Scoring Alternatif dengan XAI

Notebook ini mengimplementasikan seluruh 5 layer arsitektur dari `credit_scoring_xai.md` (Bab III & IV) dan dapat dijalankan langsung di **Google Colab** maupun lingkungan **Python lokal**.

### Lapisan yang Diuji:
1. **Layer 1:** Data Ingestion, UU PDP Consent Gateway, Tokenisasi PII SHA-256, Quality Gatekeeper (`missing > 25%`), & Feature Store.
2. **Layer 2:** Pre-processing Reweighing, Dual-Track Modeling (Baseline Scorecard WoE vs Unconstrained GBDT vs Fairness-Aware Monotonic GBDT + Platt Scaling), & Threshold Optimizer.
3. **Layer 3:** Global SHAP, Sync TreeSHAP (`< 50 ms`), Async Calibrated LIME Cross-Check (`Top-4 Rank Stability >= 85%`), & Adverse Action Reason Generator.
4. **Layer 4:** PSI/CSI Drift Monitor, Circuit-Breaker Auto-Fallback (`PSI > 0.25`), & Immutable WORM Hash-Chain Audit Trail.
5. **Layer 5:** Underwriter HITL Cockpit (`PD < 5%`, `5%–25%`, `> 25%`), Laporan Explainability Nasabah, & OJK Governance Checklist.

In [ ]:
# [Cell 1] Setup Dependensi untuk Google Colab / Lokal
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q xgboost lightgbm shap lime fairlearn seaborn openpyxl
    print('Google Colab environment ready.')
else:
    print('Local Python environment:', sys.version)

In [ ]:
# [Cell 2] Setup Path Proyek & Dataset (Google Drive / Lokal)
from pathlib import Path

if IN_COLAB:
    # Opsi 1: Mount Google Drive tempat Anda mengunggah folder credit_scoring_xai
    # from google.colab import drive
    # drive.mount('/content/drive')
    # PROJECT_DIR = Path('/content/drive/MyDrive/credit_scoring_xai')
    PROJECT_DIR = Path('/content/credit_scoring_xai')
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('PROJECT_DIR:', PROJECT_DIR)
print('Data files:', [p.name for p in (PROJECT_DIR / 'data').glob('*')] if (PROJECT_DIR / 'data').exists() else 'Folder data/ belum ditemukan')

In [ ]:
# [Cell 3] Eksekusi End-to-End Skenario S1 - S8
from src.evaluation import run_full_poc_evaluation

summary = run_full_poc_evaluation(hc_max_rows=80000, output_dir=PROJECT_DIR / 'outputs')
summary

In [ ]:
# [Cell 4] Tampilkan Tabel Hasil Evaluasi (Layer 1 - Layer 5)
import pandas as pd
from IPython.display import display, Markdown

out_dir = PROJECT_DIR / 'outputs'
tables = [
    ('Layer 1: Governance & Gatekeeper', 'table_layer1_governance.csv'),
    ('Layer 2: Perbandingan Model & Fairness (S2 & S3)', 'table_model_comparison.csv'),
    ('Layer 3: Benchmark Latensi Real-Time & Async (S4)', 'table_latency_benchmark.csv'),
    ('Layer 3: Stabilitas Rank Top-4 SHAP vs LIME (S4)', 'table_xai_stability.csv'),
    ('Layer 2/3: Uji Monotonicity & Strategic Gaming (S5)', 'table_monotonic_gaming_stress.csv'),
    ('Layer 4: Simulasi Drift & Circuit-Breaker Fallback (S6)', 'table_drift_circuit_breaker.csv'),
    ('Operasional: Biaya Komputasi & Kelayakan RAG vs Kamus (S8)', 'table_cost_and_rag_comparison.csv'),
    ('Layer 5: OJK & UU PDP Governance Checklist', 'table_ojk_governance_checklist.csv'),
    ('Evaluasi Akhir: Matriks Pembuktian Asumsi & Revisi Arsitektur', 'table_hypothesis_verification.csv'),
]
for title, fname in tables:
    display(Markdown(f'### {title}'))
    display(pd.read_csv(out_dir / fname))

In [ ]:
# [Cell 5] Tampilkan Contoh Laporan Explainability Nasabah & Log Audit WORM
display(Markdown((out_dir / 'sample_customer_explainability_report.md').read_text(encoding='utf-8')))
print('\nContoh 2 baris pertama WORM Hash-Chain Audit Trail:')
with open(out_dir / 'audit_trail_worm.jsonl', 'r', encoding='utf-8') as f:
    for _ in range(2):
        print(f.readline()[:320] + '...')

In [ ]:
# [Cell 6] Tampilkan 5 Grafik Hasil Evaluasi
from IPython.display import Image

figs = [
    'fig1_roc_ks_calibration.png',
    'fig2_fairness_tradeoff.png',
    'fig3_latency_distribution.png',
    'fig4_xai_global_local_stability.png',
    'fig5_drift_psi_circuit_breaker.png',
]
for fg in figs:
    display(Markdown(f'### {fg}'))
    display(Image(filename=str(out_dir / fg)))